# DAgger Training Notebook

Two-stage DAgger (Dataset Aggregation) pipeline for the student
policy, addressing two distinct failure modes of plain offline BC.

## Method recap

BC fits the student on `(s, a)` pairs sampled from the experts'
state distribution. At deployment, the student visits states the
experts never see, and small action errors compound into trajectory
divergence (the classical BC distribution shift). DAgger fixes this
by *interleaving* student rollouts with expert labeling:

1. Student rolls out in the env  →  visits its own (possibly
   off-distribution) states.
2. The expert labels those states with the correct action.
3. New (state, expert_action) pairs are aggregated into the dataset.
4. The student is retrained on the larger dataset.
5. Repeat for N rounds.

Each round the student improves because it learns to **correct its
own mistakes**, not just to imitate the expert on the expert's own
states.

> Ross et al., "A Reduction of Imitation Learning and Structured
> Prediction to No-Regret Online Learning", AISTATS 2011.

## Stage 1 — Standard DAgger (3 rounds, parallel collection)

For each trained stiffness E in {5e6, 7.5e6, 1e7, 2e7}:
  - Roll out the student in 8 parallel envs (SubprocVecEnv)
  - Label visited states with the **matching** expert
  - Aggregate, retrain student

beta_schedule = [0.1, 0.0, 0.0]: round 1 mixes 10% expert / 90%
student rollouts (warm-start); subsequent rounds are pure student
(true on-policy).

## Stage 2 — Targeted DAgger (2 rounds, β=0)

Extends the dataset to **interpolated E values** {6.5e6, 9e6, 1.5e7}
that were never seen during expert training:
  - Roll out at the interpolated stiffness (physical rod actually has
    that E)
  - Label with the **nearest trained expert** (e.g. 6.5e6 -> labeled
    by the 5e6 expert; 9e6 -> labeled by the 1e7 expert)
  - Forces the student to learn smooth interpolation across E

Beta=0 because by this point the student is already competent and we
want pure on-policy data on the interpolated regimes.

## Final pipeline

The final selected model is the Targeted DAgger **Round 1**
checkpoint (`results_dagger_targeted/round_1/student_targeted_r1.pth`).
Round 2 was tested but over-specialized; Round 1 generalizes better.

## Performance notes

- Parallel env collection (8 envs) cuts data collection ~5x vs
  sequential.
- Expert batch prediction (`get_actions_and_logits_batch`) labels
  all parallel obs in one forward pass.
- Dataset size capped at 6M samples (older samples randomly
  subsampled; all new samples retained) to stay within GPU memory
  during training.

Expected total runtime: ~1.5-2h for both stages on a single GPU.


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import json
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import TensorDataset, DataLoader, random_split
from tqdm.auto import tqdm

from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecNormalize

from rod_tracking_env import RodTrackingEnv
from training_student import StudentPolicy


## Shared configuration

Paths to the trained experts, env parameters, and training
hyperparameters used across both DAgger stages. Stage-specific
overrides (epochs, learning rate, beta schedule, etc.) live in their
respective cells below.


In [ ]:
EXPERTS = {
    5e6:   {"model":   "results_expert_E5e6_v27/best_model.zip",
            "vecnorm": "results_expert_E5e6_v27/vecnorm_best.pkl"},
    7.5e6: {"model":   "results_expert_E7.5e6_v27/best_model.zip",
            "vecnorm": "results_expert_E7.5e6_v27/vecnorm_best.pkl"},
    1e7:   {"model":   "results_expert_E1e7_v27/best_model.zip",
            "vecnorm": "results_expert_E1e7_v27/vecnorm_best.pkl"},
    2e7:   {"model":   "results_expert_E2e7_v27/best_model.zip",
            "vecnorm": "results_expert_E2e7_v27/vecnorm_best.pkl"},
}

# E normalization range
E_MIN = 5e6
E_MAX = 2e7

# Env parameters
ENV_BASE = {
    "n_elem": 20,
    "sim_dt": 2.0e-4,
    "num_steps_per_update": 7,
    "base_length": 1.0,
    "base_radius": 0.05,
    "density": 1000.0,
    "NU": 11.0,
    "n_control_points": 6,
    "alpha": 75.0,
    "max_rate_of_change_of_activation": np.inf,
    "target_v_max": 0.50,
    "boundary": (-0.35, 0.35, 0.90, 1.0, -0.35, 0.35),
    "final_time": 10.0,
    "success_threshold": 0.01,
    "w_dist":       2.0,
    "w_precision":  5.0,
    "w_progress":   1.0,
    "w_smoothness": 0.03,
    "sigma_mult":   1.5,
    "sigma_floor":  0.01,
}


E_VALUES = [5e6, 7.5e6, 1e7, 2e7]

# Dataset size cap.
MAX_DATASET_SIZE = 6_000_000

# Shared noise hyperparameters applied during the retrain step.
OBS_NOISE_STD    = 0.002    # noise on the 44 obs features
E_NORM_NOISE_STD = 0.05     # noise on the E_norm conditioning input
GRAD_CLIP        = 1.0      # max gradient norm


## Utilities


In [ ]:
def normalize_E(E: float) -> float:
    """Log-scale normalize E to [0, 1] — must match training."""
    return (np.log10(E) - np.log10(E_MIN)) / (np.log10(E_MAX) - np.log10(E_MIN))


def load_student(path: str, device: torch.device):
    """Load a StudentPolicy checkpoint into eval mode."""
    ckpt       = torch.load(path, map_location=device, weights_only=False)
    obs_dim    = ckpt["obs_dim"]
    action_dim = ckpt["action_dim"]
    model = StudentPolicy(obs_dim, action_dim).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, obs_dim, action_dim


def make_env(E: float, p_static: float, seed: int):
    """Factory returning a callable that builds a fresh RodTrackingEnv.

    Wrapped this way because SubprocVecEnv needs *picklable callables*
    that build envs in worker processes, not instantiated env objects.
    """
    def _init():
        env = RodTrackingEnv(**{**ENV_BASE,
                                "young_modulus": E,
                                "p_static":      p_static})
        env.reset(seed=seed)
        return env
    return _init


class ExpertWrapper:
    """Wraps one SAC expert with batch logit prediction.

    Loads the SAC model and its VecNormalize stats (frozen), exposes
    `get_actions_and_logits_batch(obs_batch)` which returns both the
    deterministic post-tanh action (for env stepping if needed) and
    the pre-tanh logit (clamped to +/-5) which is the training target
    for the student.

    Why the deterministic mean instead of a SAC sample
    --------------------------------------------------
    At training time SAC samples actions stochastically. For
    distillation we want repeatable, clean labels, so we use the
    actor's mean logit directly (no sampling noise). Equivalent in
    spirit to `predict(deterministic=True)` but exposes the pre-tanh
    `mean_logit` which `predict(...)` would drop.
    """

    def __init__(self, model_path: str, vecnorm_path: str,
                 young_modulus: float):
        self.E     = young_modulus
        self.model = SAC.load(model_path, device="cuda")

        env_params = {**ENV_BASE, "young_modulus": young_modulus,
                      "p_static": 0.0}
        dummy_env  = DummyVecEnv([lambda: RodTrackingEnv(**env_params)])
        self.vec_norm = VecNormalize.load(vecnorm_path, dummy_env)
        self.vec_norm.training    = False
        self.vec_norm.norm_reward = False

        self.model.policy.set_training_mode(False)

    def get_actions_and_logits_batch(self, obs_batch: np.ndarray):
        """Vectorized expert prediction.

        Parameters
        ----------
        obs_batch : (N, obs_dim) float32 — raw env observations (44 dims).

        Returns
        -------
        actions : (N, action_dim) — deterministic post-tanh actions
        logits  : (N, action_dim) — pre-tanh logits clamped to +/-5
                  (cap on the regression target; tanh(5) ~= 0.9999
                  so actions are unaffected)
        """
        norm_obs   = self.vec_norm.normalize_obs(obs_batch)
        obs_tensor = self.model.policy.obs_to_tensor(norm_obs)[0]

        with torch.no_grad():
            features = self.model.policy.actor.extract_features(
                obs_tensor, self.model.policy.actor.features_extractor,
            )
            latent_pi  = self.model.policy.actor.latent_pi(features)
            mean_logit = self.model.policy.actor.mu(latent_pi)
            actions    = torch.tanh(mean_logit)

        logits = torch.clamp(mean_logit, -5.0, 5.0)
        return actions.cpu().numpy(), logits.cpu().numpy()


## Data collection

In [ ]:
def collect_dagger_data_parallel(
        student: nn.Module,
        experts: dict,
        device: torch.device,
        beta: float,
        round_num: int,
        episode_configs: list,
        n_parallel_envs: int = 8,
):
    """Roll out the student in parallel envs and collect expert labels.

    `beta` controls the mix: each parallel env independently follows
    the expert with probability `beta`, otherwise follows the
    student. Beta is typically a small warm-start value for round 1
    (e.g. 0.1) and 0 for subsequent rounds (pure on-policy data).

    The stored (state, action_label) pairs always use the EXPERT's
    label, regardless of who actually drives the env — that's the
    DAgger trick.
    """
    all_obs, all_logit, all_action = [], [], []

    for (E_phys, E_expert, n_dyn, n_stat) in episode_configs:
        expert = experts[E_expert]
        E_norm = normalize_E(E_phys)

        for regime, n_episodes_target, p_static in [
            ("dyn",  n_dyn,  0.0),
            ("stat", n_stat, 1.0),
        ]:
            # Per-env reproducible seeds.
            base_seed = round_num * 100_000 + int(E_phys / 1e5) * 100
            envs = SubprocVecEnv(
                [make_env(E_phys, p_static, base_seed + i * 7)
                 for i in range(n_parallel_envs)],
                start_method="spawn",
            )

            episodes_done = 0
            obs           = envs.reset()
            E_norm_col    = np.full((n_parallel_envs, 1), E_norm,
                                    dtype=np.float32)

            label_tag = (f"(label by E_exp={E_expert:.1e})"
                         if E_phys != E_expert else "")
            pbar = tqdm(total=n_episodes_target,
                        desc=f"R{round_num} {regime} "
                             f"E={E_phys:.1e} {label_tag}".strip())

            while episodes_done < n_episodes_target:
                # Expert labels the current batch of obs (raw, no E_norm).
                expert_actions, expert_logits = (
                    expert.get_actions_and_logits_batch(obs.astype(np.float32))
                )

                #  Student predicts on the augmented obs (with E_norm).
                obs_aug   = np.concatenate([obs, E_norm_col], axis=1).astype(np.float32)
                obs_aug_t = torch.from_numpy(obs_aug).to(device)
                with torch.no_grad():
                    student_actions = torch.tanh(student(obs_aug_t)).cpu().numpy()

                # Mix student / expert actions per env.
                if beta > 0:
                    use_expert  = np.random.random(n_parallel_envs) < beta
                    env_actions = np.where(use_expert[:, None],
                                           expert_actions, student_actions)
                else:
                    env_actions = student_actions

                # Store: state visited by the student, action label from the expert.
                for i in range(n_parallel_envs):
                    all_obs.append(obs_aug[i].copy())
                    all_logit.append(expert_logits[i].copy())
                    all_action.append(expert_actions[i].copy())

                # Step the envs with the (mixed) actions.
                obs, _, dones, _ = envs.step(env_actions)

                # Count completed episodes.
                n_done = int(np.sum(dones))
                if n_done > 0:
                    pbar.update(min(n_done, n_episodes_target - episodes_done))
                    episodes_done += n_done

            pbar.close()
            envs.close()

    X        = np.array(all_obs,    dtype=np.float32)
    y_logit  = np.array(all_logit,  dtype=np.float32)
    y_action = np.array(all_action, dtype=np.float32)
    print(f"  Round {round_num}: collected {len(X):,} new samples")
    return X, y_action, y_logit


## Dataset size limiter


In [ ]:
def limit_dataset(X_agg, y_logit_agg, y_action_agg,
                  n_new: int, max_size: int):
    """Cap aggregated dataset at `max_size`, keeping all new samples.

    Parameters
    ----------
    X_agg, y_logit_agg, y_action_agg : aggregated arrays
                                       (new samples appended at the end)
    n_new   : how many of the trailing rows are the newly collected
              samples to be preserved
    max_size : cap; if `len(X_agg) <= max_size`, no-op
    """
    if len(X_agg) <= max_size:
        return X_agg, y_logit_agg, y_action_agg

    n_old      = len(X_agg) - n_new
    n_keep_old = max_size - n_new
    old_indices = np.random.choice(n_old, n_keep_old, replace=False)
    new_indices = np.arange(n_old, len(X_agg))
    keep = np.concatenate([old_indices, new_indices])
    keep.sort()

    print(f"  Dataset trimmed: {len(X_agg):,} -> {max_size:,} "
          f"({n_keep_old:,} old + {n_new:,} new)")
    return X_agg[keep], y_logit_agg[keep], y_action_agg[keep]


## Single-round training

One pass of supervised fine-tuning over the aggregated dataset.
Identical structure to `training_student_noise.py` but parameterized
on epochs / LR / batch size / early-stop patience so both DAgger
stages can call it with their own hyperparameter recipes.


In [ ]:
def train_one_round(
        X_full:       np.ndarray,
        y_logit_full: np.ndarray,
        student_path: str,
        save_path:    Path,
        round_num:    int,
        epochs:       int,
        learning_rate: float,
        batch_size:   int,
        early_stop_patience: int,
        num_workers:  int = 4,
        val_fraction: float = 0.15,
):
    """Fine-tune the student on (X, y_logit) for one DAgger round.

    Returns the path to the best checkpoint (by val loss).
    """
    device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    obs_dim     = X_full.shape[1] - 1
    action_dim  = y_logit_full.shape[1]

    # Start from the previous round's weights.
    model, _, _ = load_student(student_path, device)
    criterion   = nn.MSELoss()
    optimizer   = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler   = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    X_t = torch.tensor(X_full,       dtype=torch.float32)
    y_t = torch.tensor(y_logit_full, dtype=torch.float32)

    n_val   = int(len(X_t) * val_fraction)
    n_train = len(X_t) - n_val

    dataset = TensorDataset(X_t, y_t)
    train_ds, val_ds = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42),
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              pin_memory=True, num_workers=num_workers,
                              persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              pin_memory=True, num_workers=num_workers,
                              persistent_workers=True)

    best_val   = float("inf")
    best_ep    = 0
    early_stop = 0

    print(f"\n  Training round {round_num}: "
          f"{n_train:,} train, {n_val:,} val, "
          f"batch={batch_size}, LR={learning_rate}")

    for epoch in range(epochs):
        # train
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx = bx.to(device, non_blocking=True)
            by = by.to(device, non_blocking=True)

            # Noise on obs (44 dims) and on E_norm (1 dim).
            obs_noise = torch.randn(bx.shape[0], obs_dim, device=device) * OBS_NOISE_STD
            E_noise   = torch.randn(bx.shape[0], 1,       device=device) * E_NORM_NOISE_STD
            noisy_E   = torch.clamp(bx[:, obs_dim:obs_dim + 1] + E_noise, 0.0, 1.0)
            bx_noisy  = torch.cat([bx[:, :obs_dim] + obs_noise, noisy_E], dim=1)

            optimizer.zero_grad()
            pred = model(bx_noisy)
            loss = criterion(pred, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item()

        avg_train = train_loss / len(train_loader)
        scheduler.step()

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx = bx.to(device, non_blocking=True)
                by = by.to(device, non_blocking=True)
                val_loss += criterion(model(bx), by).item()
        avg_val = val_loss / len(val_loader)

        if avg_val < best_val:
            best_val   = avg_val
            best_ep    = epoch + 1
            early_stop = 0
            torch.save({
                "epoch":            epoch + 1,
                "model_state_dict": model.state_dict(),
                "val_loss":         best_val,
                "obs_dim":          obs_dim,
                "action_dim":       action_dim,
                "loss_type":        "logit_mse",
                "dagger_round":     round_num,
            }, save_path)
        else:
            early_stop += 1

        if (epoch + 1) % 5 == 0:
            lr = optimizer.param_groups[0]["lr"]
            print(f"    Ep {epoch + 1:>3d}/{epochs} | "
                  f"Train: {avg_train:.5f} | Val: {avg_val:.5f} | "
                  f"LR: {lr:.1e}")

        if early_stop >= early_stop_patience:
            print(f"    Early stop at ep {epoch + 1} "
                  f"(best={best_val:.5f} @ ep {best_ep})")
            break

    print(f"  Round {round_num} best: ep {best_ep}, val={best_val:.5f}")
    return str(save_path)


## Quick validation


In [ ]:
def quick_validate(student_path: str, device: torch.device,
                   E_values: list, n_episodes: int = 10):
    """Evaluate the student at a list of E values; return per-E metrics."""
    model, _, _ = load_student(student_path, device)
    results = {}

    for E in E_values:
        E_norm = normalize_E(E)
        env    = RodTrackingEnv(**{**ENV_BASE,
                                   "young_modulus": E,
                                   "p_static":      0.0})

        all_errors = []
        ep_on_goal = []

        for ep in range(n_episodes):
            obs, _ = env.reset(seed=ep * 7919)   # prime seed multiplier
            done = False
            ep_err = []
            while not done:
                obs_aug = np.concatenate([obs, [E_norm]]).astype(np.float32)
                with torch.no_grad():
                    x      = torch.from_numpy(obs_aug).unsqueeze(0).to(device)
                    action = torch.tanh(model(x)).squeeze(0).cpu().numpy()
                obs, _, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                ep_err.append(info["error"])
            all_errors.extend(ep_err)
            ep_on_goal.append(info["on_goal_fraction"])

        env.close()
        results[E] = {
            "@1cm":   float(np.mean(np.array(all_errors) < 0.01)) * 100,
            "@2cm":   float(np.mean(np.array(all_errors) < 0.02)) * 100,
            "OnGoal": float(np.mean(ep_on_goal)),
        }
    return results


def print_quick_validation(results: dict, title: str = ""):
    """Pretty-print the dict returned by `quick_validate`."""
    if title:
        print(f"\n  {title}")
    print(f"   {'E':<12} {'@1cm':>8} {'@2cm':>8} {'OnGoal':>8}")
    for E, r in results.items():
        print(f"   {E:<12.1e} {r['@1cm']:>7.1f}% "
              f"{r['@2cm']:>7.1f}% {r['OnGoal']:>7.3f}")


## Stage 1 — Standard DAgger

3 rounds on the 4 trained stiffness values. Starts from the
v4_noise student.


In [ ]:
# tage 1 config
S1_INITIAL_STUDENT = "results_student_v4_noise/student_policy.pth"
S1_ORIGINAL_DATA   = "distillation_data/distillation_dataset.npz"
S1_OUTPUT_DIR      = Path("results_dagger_fast")

# Episode configs: same E for both physical and labeling expert.
# 40 dynamic + 20 static episodes per expert per round.
S1_EPISODE_CONFIGS = [(E, E, 40, 20) for E in EXPERTS]

# Number of DAgger rounds and per-round beta.
S1_ROUNDS        = 3
S1_BETA_SCHEDULE = [0.1, 0.0, 0.0]

# Per-round training hyperparameters (slightly more aggressive than
# Stage 2 because we're still moving away from the BC initial point).
S1_TRAIN_KWARGS = dict(
    epochs              = 40,
    learning_rate       = 2e-4,
    batch_size          = 1024,
    early_stop_patience = 10,
    num_workers         = 4,
)

# Parallel collection envs (8 = good speed/RAM tradeoff for this rod
# size; tune down if running out of memory).
S1_N_PARALLEL = 8


In [ ]:
def main_dagger():
    """Stage 1 driver: standard DAgger over the 4 trained experts."""
    S1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print("\nLoading original distillation dataset")
    orig = np.load(S1_ORIGINAL_DATA)
    X_agg        = orig["X"].copy()
    y_logit_agg  = orig["y_logit"].copy()
    y_action_agg = orig.get("y_action", orig.get("y")).copy()
    print(f"   Original dataset: {X_agg.shape[0]:,} samples")

    print("\nLoading experts")
    experts = {}
    for E, cfg in EXPERTS.items():
        experts[E] = ExpertWrapper(cfg["model"], cfg["vecnorm"], E)
        print(f"   E={E:.0e} loaded")

    current_student_path = S1_INITIAL_STUDENT

    print("\nBaseline validation (v4_noise student)")
    baseline = quick_validate(current_student_path, device, E_VALUES)
    print_quick_validation(baseline)
    all_results = {"baseline": baseline}

    for round_num in range(1, S1_ROUNDS + 1):
        beta = S1_BETA_SCHEDULE[min(round_num - 1, len(S1_BETA_SCHEDULE) - 1)]

        print(f"\n{'=' * 60}")
        print(f"  STAGE 1 — ROUND {round_num}/{S1_ROUNDS}  (beta={beta:.3f})")
        print(f"{'=' * 60}")

        # Load the current student for rollout.
        student, _, _ = load_student(current_student_path, device)

        # Collect new (state, expert_action) pairs.
        X_new, y_a_new, y_l_new = collect_dagger_data_parallel(
            student, experts, device, beta, round_num,
            episode_configs=S1_EPISODE_CONFIGS,
            n_parallel_envs=S1_N_PARALLEL,
        )

        # Aggregate and trim.
        X_agg        = np.concatenate([X_agg,        X_new],   axis=0)
        y_logit_agg  = np.concatenate([y_logit_agg,  y_l_new], axis=0)
        y_action_agg = np.concatenate([y_action_agg, y_a_new], axis=0)
        X_agg, y_logit_agg, y_action_agg = limit_dataset(
            X_agg, y_logit_agg, y_action_agg,
            n_new=len(X_new), max_size=MAX_DATASET_SIZE,
        )
        print(f"  Current dataset: {X_agg.shape[0]:,} samples")

        # Snapshot dataset.
        np.savez(
            S1_OUTPUT_DIR / f"dataset_dagger_r{round_num}.npz",
            X=X_agg, y_action=y_action_agg, y_logit=y_logit_agg,
        )

        # Train next student.
        round_dir = S1_OUTPUT_DIR / f"round_{round_num}"
        round_dir.mkdir(exist_ok=True)
        save_path = round_dir / f"student_dagger_r{round_num}.pth"
        current_student_path = train_one_round(
            X_agg, y_logit_agg, current_student_path,
            save_path=save_path, round_num=round_num,
            **S1_TRAIN_KWARGS,
        )

        # Quick validation.
        results = quick_validate(current_student_path, device, E_VALUES)
        print_quick_validation(results, title=f"Round {round_num} validation:")
        all_results[f"round_{round_num}"] = results

    # Final pointer for downstream stages.
    final_path = S1_OUTPUT_DIR / "student_dagger_final.pth"
    shutil.copy(current_student_path, final_path)
    print(f"\nFinal model: {final_path}")

    # Cross-round summary table.
    print(f"\n{'=' * 70}")
    print("  STAGE 1 SUMMARY — @1cm by round")
    print(f"{'=' * 70}")
    header = f"{'Round':<12} " + "".join(f"E={E:.0e} @1cm  " for E in E_VALUES)
    print(header)
    for label, res in all_results.items():
        line = f"{label:<12} " + "".join(
            f"    {res[E]['@1cm']:>5.1f}%    " for E in E_VALUES)
        print(line)

    with open(S1_OUTPUT_DIR / "dagger_summary.json", "w") as f:
        json.dump(all_results, f, indent=2, default=float)
    print(f"\nSummary saved to {S1_OUTPUT_DIR}/dagger_summary.json")


main_dagger()


## Stage 2 — Targeted DAgger

Adds 2 rounds of DAgger on **interpolated E values** to force smooth
behavior between the 4 trained experts. Labels for interpolated
points come from the *nearest* trained expert (e.g. 6.5e6 -> labeled
by 5e6 expert; 9e6 -> labeled by 1e7 expert).

Starts from the Stage 1 final student.

**Note**: in our experiments, **Round 1 was selected as the final
model** (`results_dagger_targeted/round_1/student_targeted_r1.pth`).
Round 2 over-specialized on the interpolation regimes and regressed
slightly on the trained experts. Run Round 2 only if you want to
inspect that effect.


In [ ]:
# Stage 2 config
S2_INITIAL_STUDENT = "results_dagger_fast/student_dagger_final.pth"
S2_OUTPUT_DIR      = Path("results_dagger_targeted")

# Episode configs: list of (E_phys, E_expert, n_dyn, n_stat).
# Interpolated stiffness values are labeled by the NEAREST trained
# expert in log-E space.
S2_EPISODE_CONFIGS = [
    (5e6,    5e6,   25, 10),   # trained expert
    (6.5e6,  5e6,   25, 10),   # interpolation 5 <-> 7.5 -> labeled by 5e6
    (7.5e6,  7.5e6, 25, 10),   # trained expert
    (9e6,    1e7,   25, 10),   # interpolation 7.5 <-> 10 -> labeled by 1e7
    (1e7,    1e7,   25, 10),   # trained expert
    (1.5e7,  2e7,   25, 10),   # interpolation 10 <-> 20 -> labeled by 2e7
    (2e7,    2e7,   25, 10),   # trained expert
]

# 2 rounds suffice — we're already in fine-tuning territory.
S2_ROUNDS = 2
S2_BETA   = 0.0    # pure on-policy: the student is already competent

# Slightly less aggressive training hyperparameters than Stage 1 —
# this is fine-tuning ON TOP of fine-tuning, so we step gently.
S2_TRAIN_KWARGS = dict(
    epochs              = 30,
    learning_rate       = 1e-4,
    batch_size          = 1024,
    early_stop_patience = 8,
    num_workers         = 4,
)

S2_N_PARALLEL = 8

# Validation E values include the interpolation points so we can
# track generalization improvements directly.
S2_E_VALUES_VAL = [5e6, 6.5e6, 7.5e6, 9e6, 1e7, 1.5e7, 2e7]


In [ ]:
def main_targeted():
    """Stage 2 driver: Targeted DAgger over experts + interpolations."""
    S2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Initial student: {S2_INITIAL_STUDENT}")
    print(f"Output dir     : {S2_OUTPUT_DIR}")

    # Start from the latest Stage 1 dataset snapshot if present;
    #    fall back to the original distillation dataset otherwise.
    print("\n1. Loading base dataset...")
    s1_snapshots = sorted(Path("results_dagger_fast").glob("dataset_dagger_r*.npz"))
    if s1_snapshots:
        ds_path = s1_snapshots[-1]
        print(f"   Using Stage 1 snapshot: {ds_path}")
    else:
        ds_path = Path("distillation_data/distillation_dataset.npz")
        print(f"   No Stage 1 snapshot found — falling back to {ds_path}")
    orig = np.load(ds_path)
    X_agg        = orig["X"].copy()
    y_logit_agg  = orig["y_logit"].copy()
    y_action_agg = orig.get("y_action", orig.get("y")).copy()
    print(f"   Base dataset: {X_agg.shape[0]:,} samples")

    # Load experts (same as Stage 1).
    print("\n2. Loading experts...")
    experts = {}
    for E, cfg in EXPERTS.items():
        experts[E] = ExpertWrapper(cfg["model"], cfg["vecnorm"], E)
        print(f"   E={E:.0e} loaded")

    current_student_path = S2_INITIAL_STUDENT

    # Baseline validation including interpolation points.
    print("\n3. Baseline validation (Stage 1 final student)...")
    baseline = quick_validate(current_student_path, device, S2_E_VALUES_VAL)
    print_quick_validation(baseline)
    all_results = {"baseline": baseline}

    # Targeted rounds.
    for round_num in range(1, S2_ROUNDS + 1):
        print(f"\n{'=' * 60}")
        print(f"  STAGE 2 — TARGETED ROUND {round_num}/{S2_ROUNDS}  "
              f"(beta={S2_BETA})")
        print(f"{'=' * 60}")

        student, _, _ = load_student(current_student_path, device)

        # Round_num+10 offsets the seed scheme away from Stage 1 seeds.
        X_new, y_a_new, y_l_new = collect_dagger_data_parallel(
            student, experts, device, S2_BETA, round_num + 10,
            episode_configs=S2_EPISODE_CONFIGS,
            n_parallel_envs=S2_N_PARALLEL,
        )

        # Aggregate and trim.
        X_agg        = np.concatenate([X_agg,        X_new],   axis=0)
        y_logit_agg  = np.concatenate([y_logit_agg,  y_l_new], axis=0)
        y_action_agg = np.concatenate([y_action_agg, y_a_new], axis=0)
        X_agg, y_logit_agg, y_action_agg = limit_dataset(
            X_agg, y_logit_agg, y_action_agg,
            n_new=len(X_new), max_size=MAX_DATASET_SIZE,
        )
        print(f"  Current dataset: {X_agg.shape[0]:,} samples")

        # Snapshot.
        np.savez(
            S2_OUTPUT_DIR / f"dataset_targeted_r{round_num}.npz",
            X=X_agg, y_action=y_action_agg, y_logit=y_logit_agg,
        )

        # Train.
        round_dir = S2_OUTPUT_DIR / f"round_{round_num}"
        round_dir.mkdir(exist_ok=True)
        save_path = round_dir / f"student_targeted_r{round_num}.pth"
        current_student_path = train_one_round(
            X_agg, y_logit_agg, current_student_path,
            save_path=save_path, round_num=round_num,
            **S2_TRAIN_KWARGS,
        )

        # Quick validation INCLUDING interpolation points — this is
        # where Stage 2 should show clear improvement vs Stage 1.
        results = quick_validate(current_student_path, device, S2_E_VALUES_VAL)
        print_quick_validation(results, title=f"Round {round_num} validation:")
        all_results[f"round_{round_num}"] = results

    # Final pointer.
    final_path = S2_OUTPUT_DIR / "student_targeted_final.pth"
    shutil.copy(current_student_path, final_path)
    print(f"\nFinal model: {final_path}")
    print(f"NOTE: in our experiments, round_1 was preferred over the "
          f"final (round 2) model because round 2 over-specialized "
          f"on the interpolation regimes.")

    # Cross-round summary including interpolation columns.
    print(f"\n{'=' * 80}")
    print("  STAGE 2 SUMMARY — @1cm by round (including interpolations)")
    print(f"{'=' * 80}")
    print(f"{'Round':<12}" + "".join(f"  E={E:.1e}" for E in S2_E_VALUES_VAL))
    print("-" * 80)
    for label, res in all_results.items():
        print(f"{label:<12}" + "".join(
            f"   {res[E]['@1cm']:>5.1f}%" for E in S2_E_VALUES_VAL))

    with open(S2_OUTPUT_DIR / "targeted_summary.json", "w") as f:
        json.dump(all_results, f, indent=2, default=float)
    print(f"\nSummary saved to {S2_OUTPUT_DIR}/targeted_summary.json")


main_targeted()
